# Reserve price prediction with CLM-8B fine-tuning (POC)

Predicts a **daily reserve (minimum) price band per SKU × warehouse** by fine-tuning
[CLM-8B](https://github.com/Contrastive-LM/CLM) (frozen Qwen3-8B reader + small trained heads),
and scores it against Assurant's own historical floors.

**What the label is.** On a day where the same SKU × warehouse had both an accepted and a
non-accepted (countered / rejected) offer, Assurant's floor lies between the highest bid it turned
down and the lowest bid it accepted. The model learns to place today's reserve in that interval,
so a perfect score means *matches Assurant's current floor*, not *optimal price*.

**Inputs (all from days before the target day):** product attributes, a 30-day market reference
price, bids (counts, buyers, units, median/top bid vs market, accepted share), sales velocity,
Assurant's previous floor, weekday. Inventory and Buyer Profile files are not used (no date
overlap / label leakage).

**Candidates:** 14 reserve bands (multiples of the market price) described in English.
**Training:** CLM's soft-target InfoNCE on the heads, warm-started from the released CLM head,
3 seeds, early stopping on February validation. **Test:** March–April 2026.

**Runtime:** a GPU with ≥ 24 GB (L4 / A10G / A100) or 2 × 16 GB GPUs. Qwen3-8B needs ~16 GB.
Embedding ~35K texts takes roughly 10–25 minutes depending on the GPU; training the heads takes minutes.

**Data:** `Offers`, `Product Information` and `Sales Order&Line items` (`.csv` or `.xlsx`; CSV is used
when both exist and loads much faster) anywhere under `DATA_DIR` = `/dist_home/suryansh/sharukesh/Work`.
**Outputs** (embedding cache, trained heads, result tables) go to `DATA_DIR/reserve_clm_run`, so they
survive restarts: re-running the notebook reuses the cached embeddings instead of embedding again.


In [ ]:
#@title 1. Environment setup
import os, subprocess, sys
CLM_COMMIT = "d5f9ef0fd9bde185df0ceaad4f4ecc6cfe8c34f6"
if os.environ.get("RESERVE_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.51", "accelerate",
                    "pandas", "pyarrow", "openpyxl", "huggingface_hub"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    f"contrastive-lm @ git+https://github.com/Contrastive-LM/CLM@{CLM_COMMIT}"], check=True)
import torch
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "none")


In [ ]:
#@title 2. Data and output folders (persistent server storage)
DATA_DIR = os.environ.get("RESERVE_DATA_DIR", "/dist_home/suryansh/sharukesh/Work")
WORK = os.environ.get("RESERVE_WORK_DIR", os.path.join(DATA_DIR, "reserve_clm_run"))
assert os.path.isdir(DATA_DIR), f"data folder not found: {DATA_DIR}"
DATA_ROOTS = [DATA_DIR]
os.makedirs("reserve_clm", exist_ok=True)
os.makedirs(WORK, exist_ok=True)
print("data from", DATA_DIR, "| outputs ->", WORK)


## 3. Project code
The tested modules, written to `reserve_clm/` (collapsed).

In [ ]:
%%writefile reserve_clm/__init__.py
"""Reserve-price band prediction with CLM-8B fine-tuning (POC)."""


In [ ]:
%%writefile reserve_clm/config.py
"""Project-wide constants for the reserve-price CLM POC."""
import pandas as pd

OFFER_STATUSES = ("ACCEPTED", "COUNTERED", "NOT_ACCEPTED")
TRAIN_END = pd.Timestamp("2026-02-01")   # train: day < TRAIN_END
VAL_END = pd.Timestamp("2026-03-01")     # val: TRAIN_END <= day < VAL_END; test: day >= VAL_END
REF_WINDOW_DAYS = 30
TREND_WINDOW_DAYS = 7
MIN_REF_ACCEPTS = 3                      # accepted offers needed in the window for a reference price
CLM_REPO_COMMIT = "d5f9ef0fd9bde185df0ceaad4f4ecc6cfe8c34f6"


In [ ]:
%%writefile reserve_clm/bands.py
"""Reserve bands: multiples of the reference (market) price that CLM chooses between."""
import numpy as np

INNER_EDGES = np.round(np.arange(0.80, 1.10001, 0.025), 3)       # 0.800 ... 1.100 (13 edges)
EDGES = np.concatenate([[0.70], INNER_EDGES, [1.20]])             # outer edges bound the tails
N_BANDS = len(EDGES) - 1                                          # 14
BAND_KEYS = [f"b{i:02d}" for i in range(N_BANDS)]
BAND_LABELS = (["<0.80x"] + [f"{EDGES[i]:.3f}-{EDGES[i + 1]:.3f}x" for i in range(1, N_BANDS - 1)]
               + [">=1.10x"])
BAND_MIDS = (EDGES[:-1] + EDGES[1:]) / 2


def band_of(ratio) -> np.ndarray:
    """Band index (0..13) of each ratio; anything below 0.80 is band 0, from 1.10 up band 13."""
    r = np.asarray(ratio, dtype=float)
    return np.searchsorted(EDGES[1:-1], r, side="right")


def soft_targets(lo, hi) -> np.ndarray:
    """Distribution over bands proportional to the overlap with [lo, hi] (ratios).

    A zero-width interval puts all mass on the band containing it.
    """
    lo = np.clip(np.atleast_1d(np.asarray(lo, dtype=float)), EDGES[0], EDGES[-1])
    hi = np.clip(np.atleast_1d(np.asarray(hi, dtype=float)), EDGES[0], EDGES[-1])
    a, b = EDGES[:-1][None, :], EDGES[1:][None, :]
    overlap = np.clip(np.minimum(hi[:, None], b) - np.maximum(lo[:, None], a), 0.0, None)
    width = overlap.sum(1)
    out = np.zeros_like(overlap)
    wide = width > 1e-12
    out[wide] = overlap[wide] / width[wide, None]
    point = ~wide
    out[point, band_of((lo[point] + hi[point]) / 2)] = 1.0
    return out


In [ ]:
%%writefile reserve_clm/data.py
"""Locate and load the three source files, keeping only what the POC trusts."""
from __future__ import annotations

import glob
import os

import pandas as pd

from .config import OFFER_STATUSES

FILES = {
    "offers": "Offers.csv",
    "products": "Product Information.csv",
    "sales": "Sales Order&Line items.csv",
}
_TS = "%m/%d/%Y %H:%M"


def _norm(name: str) -> str:
    """File name without extension, with case, spaces and punctuation ignored (uploads often drop '&')."""
    stem = os.path.splitext(name)[0]
    return "".join(ch for ch in stem.lower() if ch.isalnum())


def find_data_files(roots: list[str]) -> dict[str, str]:
    """First match of each required file under the given roots (searched recursively, in order).

    Accepts .csv or .xlsx (CSV preferred when both exist). Names are compared ignoring case,
    spaces and punctuation, so 'Sales Order Line items.csv' also matches 'Sales Order&Line items.csv'.
    Excel lock files ('~$...') are ignored.
    """
    wanted = {name: _norm(fname) for name, fname in FILES.items()}
    out: dict[str, str] = {}
    for root in roots:
        if not root or not os.path.isdir(root) or len(out) == len(FILES):
            continue
        for ext in ("csv", "xlsx"):
            paths = glob.glob(os.path.join(glob.escape(root), "**", f"*.{ext}"), recursive=True)
            for path in sorted(paths):
                base = os.path.basename(path)
                if base.startswith("~$"):
                    continue
                for name, key in wanted.items():
                    if name not in out and _norm(base) == key:
                        out[name] = path
    missing = [FILES[k] for k in FILES if k not in out]
    if missing:
        raise FileNotFoundError(f"could not find {missing} (.csv or .xlsx) under {roots}")
    return out


def _read(path, cols) -> pd.DataFrame:
    """CSV or Excel, every column as text."""
    if isinstance(path, str) and path.lower().endswith(".xlsx"):
        return pd.read_excel(path, dtype=str, usecols=cols)
    return pd.read_csv(path, dtype=str, usecols=cols, encoding_errors="ignore")


def _id(s: pd.Series) -> pd.Series:
    """Ids read from Excel can come back as '12345.0'; make them match the CSV form '12345'."""
    return s.str.strip().str.replace(r"\.0$", "", regex=True)


def _ts(s: pd.Series) -> pd.Series:
    """CSV timestamps are 'MM/DD/YYYY HH:MM'; Excel ones come back as ISO text. Accept both."""
    out = pd.to_datetime(s, format=_TS, errors="coerce")
    retry = out.isna() & s.notna()
    if retry.any():
        out[retry] = pd.to_datetime(s[retry], format="mixed", errors="coerce")
    return out.astype("datetime64[ns]")


def _day(s: pd.Series) -> pd.Series:
    return _ts(s).dt.normalize()


def load_offers(path) -> pd.DataFrame:
    cols = [
        "OFFER_ID",
        "BUYER_ID",
        "SKU",
        "WAREHOUSE",
        "OFFER_QUANTITY",
        "OFFER_PRICE",
        "OFFER_STATUS",
        "OFFER_CREATE_DATETIME",
    ]
    df = _read(path, cols)
    out = pd.DataFrame({
        "offer_id": _id(df.OFFER_ID),
        "buyer_id": _id(df.BUYER_ID),
        "sku": _id(df.SKU),
        "wh": _id(df.WAREHOUSE),
        "qty": pd.to_numeric(df.OFFER_QUANTITY, errors="coerce").astype(float),
        "price": pd.to_numeric(df.OFFER_PRICE, errors="coerce").astype(float),
        "status": df.OFFER_STATUS,
        "ts": _ts(df.OFFER_CREATE_DATETIME),
    })
    out = out[out.status.isin(OFFER_STATUSES) & (out.price > 0) & out.ts.notna()].copy()
    out["day"] = out.ts.dt.normalize().astype("datetime64[ns]")
    out["accepted"] = out.status.eq("ACCEPTED")
    out["qty"] = out.qty.where(out.qty > 0, 1.0).astype(float)
    return out.reset_index(drop=True)


def load_sales(path) -> pd.DataFrame:
    cols = [
        "WAREHOUSE_CODE",
        "STATUS",
        "ORDER_TYPE",
        "DATE",
        "PRODUCT_ID",
        "UNIT_PRICE",
        "SALES_QTY",
        "QUANTITY_CANCELLED",
    ]
    df = _read(path, cols)
    units = (
        pd.to_numeric(df.SALES_QTY, errors="coerce").fillna(0)
        - pd.to_numeric(df.QUANTITY_CANCELLED, errors="coerce").fillna(0)
    ).astype(float)
    out = pd.DataFrame({
        "sku": _id(df.PRODUCT_ID),
        "wh": _id(df.WAREHOUSE_CODE),
        "status": df.STATUS,
        "order_type": df.ORDER_TYPE,
        "units": units,
        "unit_price": pd.to_numeric(df.UNIT_PRICE, errors="coerce").astype(float),
        "day": _day(df.DATE),
    })
    return out[(out.status != "CANCELED") & (out.units > 0) & out.day.notna()].reset_index(drop=True)


def load_products(path) -> pd.DataFrame:
    cols = {
        "SKU": "sku",
        "WAREHOUSE": "wh",
        "MANUFACTURER": "manufacturer",
        "MODEL": "model",
        "CAPACITY": "capacity",
        "GRADE": "grade",
        "COLOR": "color",
        "CARRIER": "carrier",
        "LOCK_STATUS": "lock_status",
        "CATEGORY": "category",
    }
    df = _read(path, list(cols))
    out = df.rename(columns=cols)[list(cols.values())].copy()
    out["sku"], out["wh"] = _id(out["sku"]), _id(out["wh"])
    for c in list(cols.values())[2:]:
        out[c] = out[c].fillna("unknown").str.strip().replace("", "unknown")
    out["family"] = out.manufacturer + "|" + out.model + "|" + out.capacity + "|" + out.grade
    return out.drop_duplicates(["sku", "wh"]).reset_index(drop=True)


In [ ]:
%%writefile reserve_clm/labels.py
"""Assurant's implicit daily floor, read from OFFER_STATUS.

On a SKU x warehouse x day with both outcomes, the floor sits between the highest bid
Assurant did not accept (countered or rejected) and the lowest bid it accepted.
"""
import numpy as np
import pandas as pd

from .config import TRAIN_END, VAL_END

GKEYS = ["sku", "wh", "day"]


def floor_intervals(offers: pd.DataFrame) -> pd.DataFrame:
    if offers.empty:
        return pd.DataFrame(columns=["sku", "wh", "day", "lo", "hi", "n_offers_day", "mid"])
    lo = offers[~offers.accepted].groupby(GKEYS).price.max().rename("lo")
    hi = offers[offers.accepted].groupby(GKEYS).price.min().rename("hi")
    n = offers.groupby(GKEYS).size().rename("n_offers_day")
    df = pd.concat([lo, hi], axis=1, join="inner").join(n).reset_index()
    if df.empty:
        return pd.DataFrame(columns=["sku", "wh", "day", "lo", "hi", "n_offers_day", "mid"])
    df = df[df.lo <= df.hi].copy()
    df["mid"] = (df.lo + df.hi) / 2
    return df.reset_index(drop=True)


def assign_split(day: pd.Series) -> np.ndarray:
    day = pd.to_datetime(day)
    return np.where(day < TRAIN_END, "train", np.where(day < VAL_END, "val", "test"))


In [ ]:
%%writefile reserve_clm/features.py
"""Features for each target SKU x warehouse x day, using only days strictly before it."""
from __future__ import annotations

import numpy as np
import pandas as pd

from .config import MIN_REF_ACCEPTS, REF_WINDOW_DAYS, TREND_WINDOW_DAYS

KEYS = ["sku", "wh"]
GKEYS = ["sku", "wh", "day"]


def lagged_join(events: pd.DataFrame, targets: pd.DataFrame, keys: list[str], window: int) -> pd.DataFrame:
    """Event rows from days d-window ... d-1 for every target (keys, d); `day` is the target day."""
    t = targets[keys + ["day"]].dropna(subset=keys).drop_duplicates()
    parts = []
    for lag in range(1, window + 1):
        e = events.assign(day=events["day"] + pd.Timedelta(days=lag), lag=lag)
        parts.append(t.merge(e, on=keys + ["day"], how="inner"))
    if not parts:
        cols = list(dict.fromkeys(keys + ["day"] + list(events.columns) + ["lag"]))
        return pd.DataFrame(columns=cols)
    return pd.concat(parts, ignore_index=True)


def _window_ref(daily: pd.DataFrame, t: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    j = lagged_join(daily, t, keys, REF_WINDOW_DAYS)
    gk = keys + ["day"]
    g = j.groupby(gk)
    f = pd.DataFrame({"ref30": g.med.median(), "ref_n": g.cnt.sum()})
    f["ref7"] = j[j.lag <= TREND_WINDOW_DAYS].groupby(gk).med.median()
    return f.reset_index()


def reference_prices(offers: pd.DataFrame, products: pd.DataFrame, targets: pd.DataFrame) -> pd.DataFrame:
    """30-day median of daily median accepted prices: SKU level, else product family, else none."""
    fam = products[KEYS + ["family"]].drop_duplicates(subset=KEYS)
    acc = offers[offers["accepted"].astype(bool)].merge(fam, on=KEYS, how="left")
    sku_daily = acc.groupby(["sku", "day"]).price.agg(med="median", cnt="size").reset_index()
    fam_daily = (acc.dropna(subset=["family"]).groupby(["family", "day"]).price
                 .agg(med="median", cnt="size").reset_index())
    tt = targets[GKEYS].drop_duplicates().merge(fam, on=KEYS, how="left")
    s = _window_ref(sku_daily, tt, ["sku"]).rename(columns={"ref30": "ref30_sku", "ref_n": "ref_n_sku", "ref7": "ref7_sku"})
    f = _window_ref(fam_daily, tt, ["family"]).rename(columns={"ref30": "ref30_fam", "ref_n": "ref_n_fam", "ref7": "ref7_fam"})
    out = tt.merge(s, on=["sku", "day"], how="left").merge(f, on=["family", "day"], how="left")
    use_sku = (out.ref_n_sku.fillna(0) >= MIN_REF_ACCEPTS).to_numpy()
    use_fam = ~use_sku & (out.ref_n_fam.fillna(0) >= MIN_REF_ACCEPTS).to_numpy()
    out["ref_source"] = np.where(use_sku, "sku", np.where(use_fam, "family", "none"))
    out["ref_price"] = np.where(use_sku, out.ref30_sku, np.where(use_fam, out.ref30_fam, np.nan))
    out["ref_n"] = np.where(use_sku, out.ref_n_sku, np.where(use_fam, out.ref_n_fam, 0.0))
    ref7 = np.where(use_sku, out.ref7_sku, np.where(use_fam, out.ref7_fam, np.nan))
    out["ref_trend"] = ref7 / out["ref_price"]
    return out[GKEYS + ["family", "ref_source", "ref_price", "ref_n", "ref_trend"]]


def bid_features(offers: pd.DataFrame, t: pd.DataFrame) -> pd.DataFrame:
    daily = offers.groupby(GKEYS).agg(n=("price", "size"), qty=("qty", "sum"), acc=("accepted", "sum"),
                                      med=("price", "median"), top=("price", "max")).reset_index()
    j = lagged_join(daily, t, KEYS, REF_WINDOW_DAYS)
    g = j.groupby(GKEYS)
    f = pd.DataFrame({"n_offers_30": g.n.sum(), "units_req_30": g.qty.sum(), "acc_30": g.acc.sum(),
                      "bid_med_30": g.med.median(), "bid_top_30": g.top.max()})
    f["n_offers_7"] = j[j.lag <= TREND_WINDOW_DAYS].groupby(GKEYS).n.sum()
    bidders = offers[GKEYS + ["buyer_id"]].drop_duplicates()
    f["n_bidders_30"] = lagged_join(bidders, t, KEYS, REF_WINDOW_DAYS).groupby(GKEYS).buyer_id.nunique()
    out = t[GKEYS].merge(f.reset_index(), on=GKEYS, how="left")
    counts = ["n_offers_30", "units_req_30", "acc_30", "n_offers_7", "n_bidders_30"]
    out[counts] = out[counts].fillna(0)
    return out


def _last_before(events: pd.DataFrame, t: pd.DataFrame, name: str) -> pd.DataFrame:
    """Most recent event strictly before each target day (same sku, wh); event day renamed to `name`."""
    r = events.rename(columns={"day": name}).dropna(subset=[name]).sort_values(name)
    left = t[GKEYS].sort_values("day")
    if r.empty or left.empty:
        res = left.copy()
        for col in r.columns:
            if col not in KEYS and col != name:
                res[col] = np.nan
        res[name] = pd.NaT
        return res
    r[name] = pd.to_datetime(r[name]).astype(left["day"].dtype)
    return pd.merge_asof(left, r, left_on="day", right_on=name, by=KEYS,
                         allow_exact_matches=False, direction="backward")


def sales_features(sales: pd.DataFrame, t: pd.DataFrame) -> pd.DataFrame:
    daily = sales.groupby(GKEYS, as_index=False).units.sum()
    j = lagged_join(daily, t, KEYS, REF_WINDOW_DAYS)
    f = j.groupby(GKEYS).units.sum().rename("units_sold_30").to_frame()
    f["units_sold_7"] = j[j.lag <= TREND_WINDOW_DAYS].groupby(GKEYS).units.sum()
    stock = sales[sales.order_type == "STOCKLIST"].groupby(GKEYS, as_index=False).unit_price.median()
    f["sale_price_med_30"] = lagged_join(stock, t, KEYS, REF_WINDOW_DAYS).groupby(GKEYS).unit_price.median()
    out = t[GKEYS].merge(f.reset_index(), on=GKEYS, how="left")
    out[["units_sold_30", "units_sold_7"]] = out[["units_sold_30", "units_sold_7"]].fillna(0)
    last = _last_before(daily[GKEYS], t, "last_sale_day")
    out = out.merge(last, on=GKEYS, how="left")
    out["days_since_last_sale"] = (out.day - out.last_sale_day).dt.days
    return out.drop(columns=["last_sale_day"])


def last_floor_features(floors: pd.DataFrame, t: pd.DataFrame) -> pd.DataFrame:
    ev = floors[GKEYS + ["lo", "hi"]].rename(columns={"lo": "last_lo", "hi": "last_hi"})
    m = _last_before(ev, t, "last_floor_day")
    m["days_since_last_floor"] = (m.day - m.last_floor_day).dt.days
    return m.drop(columns=["last_floor_day"])


FEATURE_COLS = [
    "sku", "wh", "day",
    "family", "ref_source", "ref_price", "ref_n", "ref_trend",
    "n_offers_30", "units_req_30", "acc_30", "bid_med_30", "bid_top_30", "n_offers_7",
    "n_bidders_30", "units_sold_30", "units_sold_7", "sale_price_med_30",
    "days_since_last_sale", "last_lo", "last_hi", "days_since_last_floor",
    "manufacturer", "model", "capacity", "grade", "color", "carrier", "lock_status", "category",
    "bid_med_ratio_30", "bid_top_ratio_30", "sale_price_ratio_30",
    "last_lo_ratio", "last_hi_ratio", "accept_share_30",
]


def build_features(offers: pd.DataFrame, sales: pd.DataFrame, products: pd.DataFrame,
                   floors: pd.DataFrame, targets: pd.DataFrame) -> pd.DataFrame:
    t = targets[GKEYS].drop_duplicates().reset_index(drop=True)
    out = t.merge(reference_prices(offers, products, t), on=GKEYS, how="left")
    out = out.merge(bid_features(offers, t), on=GKEYS, how="left")
    out = out.merge(sales_features(sales, t), on=GKEYS, how="left")
    out = out.merge(last_floor_features(floors, t), on=GKEYS, how="left")
    out = out.merge(products.drop(columns=["family"], errors="ignore"), on=KEYS, how="left")
    for src, dst in [("bid_med_30", "bid_med_ratio_30"), ("bid_top_30", "bid_top_ratio_30"),
                     ("sale_price_med_30", "sale_price_ratio_30"), ("last_lo", "last_lo_ratio"),
                     ("last_hi", "last_hi_ratio")]:
        out[dst] = out[src] / out["ref_price"]
    out["accept_share_30"] = out.acc_30 / out.n_offers_30.where(out.n_offers_30 > 0)
    cols = [c for c in FEATURE_COLS if c in out.columns] + [c for c in out.columns if c not in FEATURE_COLS]
    return out[cols]


In [ ]:
%%writefile reserve_clm/text.py
"""What CLM reads: the state text (one SKU x warehouse x day) and the 14 band candidates."""
from __future__ import annotations

import math

import pandas as pd

from .bands import BAND_KEYS, EDGES, N_BANDS

QUESTION = ("Which reserve (minimum) price band should be set today for this SKU at this warehouse? "
            "Bands are multiples of the SKU's recent market price.")


def _int(x) -> int:
    return 0 if pd.isna(x) else int(x)


def _ratio(x) -> str:
    return "unknown" if pd.isna(x) else f"{x:.2f}x market"


def _pct(x) -> str:
    return "unknown" if pd.isna(x) else f"{x:.0%}"


def _trend(x) -> str:
    if pd.isna(x):
        return "unknown"
    pct = (x - 1) * 100
    word = "rising" if pct > 2 else "falling" if pct < -2 else "flat"
    return f"{word} ({pct:+.1f}%)"


def _demand(n: int) -> str:
    return "none" if n == 0 else "low" if n < 3 else "moderate" if n < 10 else "high"


def _velocity(u: int) -> str:
    return "no sales" if u == 0 else "slow" if u < 5 else "steady" if u < 30 else "fast"


def state_context(r) -> str:
    source = "this SKU" if r.ref_source == "sku" else "similar products (same model, capacity and grade)"
    last_sale = ("No sale recorded before today" if pd.isna(r.days_since_last_sale)
                 else f"Last sale {_int(r.days_since_last_sale)} days ago")
    if pd.isna(r.last_lo_ratio) or pd.isna(r.last_hi_ratio):
        last_floor = "No previous reserve observed for this SKU at this warehouse."
    else:
        last_floor = (f"Previous reserve between {r.last_lo_ratio:.2f}x and {r.last_hi_ratio:.2f}x market, "
                      f"{_int(r.days_since_last_floor)} days ago.")
    n7 = _int(r.n_offers_7)
    sold30 = _int(r.units_sold_30)
    return "\n".join([
        f"Product: {r.manufacturer} {r.model}, {r.capacity}, grade {r.grade}, colour {r.color}, "
        f"carrier {r.carrier}, {r.lock_status}. Warehouse {r.wh}.",
        f"Market price: ${r.ref_price:.0f} from {source}, last 30 days, {_int(r.ref_n)} accepted offers. "
        f"Trend last 7 days vs 30 days: {_trend(r.ref_trend)}.",
        f"Bids last 7 days: {n7} ({_demand(n7)} demand). Bids last 30 days: {_int(r.n_offers_30)} "
        f"from {_int(r.n_bidders_30)} buyers for {_int(r.units_req_30)} units.",
        f"Median bid {_ratio(r.bid_med_ratio_30)}, top bid {_ratio(r.bid_top_ratio_30)}. "
        f"Share of recent bids accepted: {_pct(r.accept_share_30)}.",
        f"Units sold last 7 days: {_int(r.units_sold_7)}, last 30 days: {sold30} ({_velocity(sold30)}). "
        f"{last_sale}. Median sale price {_ratio(r.sale_price_ratio_30)}.",
        last_floor,
        f"Day: {pd.Timestamp(r.day):%A}.",
    ])


def state_text(r) -> str:
    """Context first, question last: the layout CLM's heads were trained on (clm.schema.state_text)."""
    return f"{state_context(r).strip()}\n\n{QUESTION}"


_TIERS = [  # (upper bound on band midpoint, name, when to use)
    (0.875, "Heavy discount reserve", "Use when stock is slow or ageing, bids are well below market and prices are falling."),
    (0.95, "Moderate discount reserve", "Use when demand is soft, bids sit below market and sell-through is slow."),
    (1.00, "Slight discount reserve", "Use when demand is steady, several buyers bid close to market and prices are flat."),
    (1.05, "At-market reserve", "Use when demand is healthy, bids reach market price and stock sells steadily."),
    (math.inf, "Premium reserve", "Use when stock is scarce, competing bids exceed market and prices are rising."),
]


def band_descriptions() -> dict[str, str]:
    out = {}
    for i, key in enumerate(BAND_KEYS):
        lo, hi = EDGES[i], EDGES[i + 1]
        if i == 0:
            out[key] = ("Deep clearance reserve: below 0.80x the market price. Use for stale stock with very "
                        "weak, low bids and falling prices.")
        elif i == N_BANDS - 1:
            out[key] = ("Strong premium reserve: at or above 1.10x the market price. Use for scarce stock with "
                        "strong competing bids well above market and rising prices.")
        else:
            name, hint = next((n, h) for bound, n, h in _TIERS if (lo + hi) / 2 < bound)
            out[key] = f"{name}: between {lo:.3f}x and {hi:.3f}x the market price. {hint}"
    return out


In [ ]:
%%writefile reserve_clm/baselines.py
"""Rule baselines the CLM has to beat (no training)."""
from __future__ import annotations

import numpy as np
import pandas as pd

from .bands import N_BANDS


def carry_forward_ratio(df: pd.DataFrame, fallback: float) -> np.ndarray:
    """B0: midpoint of Assurant's previous floor at this SKU x warehouse, else the fixed ratio."""
    r = (df["last_lo_ratio"] + df["last_hi_ratio"]) / 2
    return r.fillna(fallback).to_numpy(dtype=float)


def fixed_ratio(n: int, ratio: float) -> np.ndarray:
    """B1/B2: one ratio for every row."""
    return np.full(n, float(ratio))


def majority_band(true_band: np.ndarray) -> int:
    return int(np.bincount(np.asarray(true_band, dtype=int), minlength=N_BANDS).argmax())


In [ ]:
%%writefile reserve_clm/evaluate.py
"""Metric primitives: interval hit, price error, band accuracy, offer replay, calibration."""
from __future__ import annotations

import numpy as np
import pandas as pd

NAN = float("nan")


def decode_bands(probs: np.ndarray, how: str = "median") -> np.ndarray:
    probs = np.asarray(probs, dtype=float)
    if how == "argmax":
        return probs.argmax(1)
    return (probs.cumsum(1) >= 0.5).argmax(1)


def interval_metrics(price, lo, hi, mid) -> dict:
    price, lo, hi, mid = (np.asarray(x, dtype=float) for x in (price, lo, hi, mid))
    keys = ["interval_hit_rate", "too_high_rate", "too_low_rate", "mape_vs_mid", "median_ape_vs_mid", "bias_vs_mid"]
    if len(price) == 0:
        return {"n": 0, **{k: NAN for k in keys}}
    rel = (price - mid) / mid
    return {
        "n": int(len(price)),
        "interval_hit_rate": float(((price >= lo) & (price <= hi)).mean()),
        "too_high_rate": float((price > hi).mean()),
        "too_low_rate": float((price < lo).mean()),
        "mape_vs_mid": float(np.abs(rel).mean()),
        "median_ape_vs_mid": float(np.median(np.abs(rel))),
        "bias_vs_mid": float(rel.mean()),
    }


def band_metrics(pred_band, true_band) -> dict:
    p, t = np.asarray(pred_band, dtype=int), np.asarray(true_band, dtype=int)
    if len(p) == 0:
        return {"band_exact": NAN, "band_within_1": NAN, "band_within_3": NAN, "band_within_5": NAN, "band_mae": NAN}
    d = np.abs(p - t)
    return {
        "band_exact": float((d == 0).mean()),
        "band_within_1": float((d <= 1).mean()),
        "band_within_3": float((d <= 3).mean()),
        "band_within_5": float((d <= 5).mean()),
        "band_mae": float(d.mean()),
    }


def replay(offers: pd.DataFrame, groups: pd.DataFrame) -> dict:
    """Use each group's reserve as that day's base price: accept every offer with bid >= reserve.

    Compared with Assurant's OFFER_STATUS per offer and per SKU x warehouse x day.
    """
    keys = ["sku", "wh", "day"]
    m = offers.merge(groups[keys + ["reserve"]], on=keys, how="inner")
    out = {"n_groups": int(len(groups)), "n_offers": int(len(m))}
    if m.empty:
        return out
    m = m.assign(pred=(m.price >= m.reserve).to_numpy(), act=m.accepted.to_numpy(dtype=bool))
    m["agree"] = m.pred == m.act
    pred, act = m.pred.to_numpy(), m.act.to_numpy()
    both = int((pred & act).sum())
    day = m.groupby(keys).agg(pred_n=("pred", "sum"), act_n=("act", "sum"), exact=("agree", "all"))
    out.update(
        agreement=float(m.agree.mean()),
        lost_sales_rate=float((~pred & act).mean()),
        underpricing_rate=float((pred & ~act).mean()),
        pred_accept_rate=float(pred.mean()),
        actual_accept_rate=float(act.mean()),
        accept_precision=both / pred.sum() if pred.any() else NAN,
        accept_recall=both / act.sum() if act.any() else NAN,
        n_days=int(len(day)),
        day_exact_match=float(day.exact.mean()),
        day_count_mae=float((day.pred_n - day.act_n).abs().mean()),
        accepted_per_day_pred=float(day.pred_n.mean()),
        accepted_per_day_actual=float(day.act_n.mean()),
    )
    return out


def calibration_table(conf, hit, n_bins: int = 5) -> pd.DataFrame:
    conf, hit = np.asarray(conf, dtype=float), np.asarray(hit, dtype=float)
    if len(conf) == 0:
        return pd.DataFrame(columns=["bin", "n", "mean_confidence", "interval_hit_rate"])
    d = pd.DataFrame({"conf": conf, "hit": hit, "bin": pd.qcut(conf, n_bins, duplicates="drop")})
    g = d.groupby("bin", observed=True)
    t = pd.DataFrame({"n": g.size(), "mean_confidence": g.conf.mean(), "interval_hit_rate": g.hit.mean()})
    t = t.reset_index()
    t["bin"] = t["bin"].astype(str)
    return t


def band_ece(probs, true_band, n_bins: int = 10) -> float:
    probs = np.asarray(probs, dtype=float)
    if len(probs) == 0:
        return NAN
    conf = probs.max(1)
    correct = probs.argmax(1) == np.asarray(true_band, dtype=int)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        m = (conf >= lo if i == 0 else conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


In [ ]:
%%writefile reserve_clm/report.py
"""Comparison tables: every method scored the same way on the same rows."""
from __future__ import annotations

import numpy as np
import pandas as pd

from .bands import BAND_LABELS, BAND_MIDS, band_of
from .baselines import carry_forward_ratio, fixed_ratio, majority_band
from .evaluate import band_metrics, decode_bands, interval_metrics, replay

KEYS3 = ["sku", "wh", "day"]


def method_ratios(df: pd.DataFrame, train_df: pd.DataFrame, runs: list[dict], split_key: str):
    """Reserve ratio (x reference price) per method for the rows of df."""
    fallback = float(train_df.mid_ratio.median())
    maj = majority_band(train_df.true_band.to_numpy())
    ratios = {"B0 carry-forward": carry_forward_ratio(df, fallback),
              "B1 fixed ratio": fixed_ratio(len(df), fallback),
              "B2 majority band": fixed_ratio(len(df), BAND_MIDS[maj])}
    for r in runs:
        ratios[f"CLM seed {r['seed']}"] = BAND_MIDS[decode_bands(r["probs"][split_key])]
    ens = np.mean([r["probs"][split_key] for r in runs], axis=0)
    ratios["CLM ensemble (median band)"] = BAND_MIDS[decode_bands(ens)]
    ratios["CLM ensemble (expected ratio)"] = ens @ BAND_MIDS
    return ratios, ens


def score_table(df: pd.DataFrame, ratios: dict[str, np.ndarray]) -> pd.DataFrame:
    rows = []
    for name, r in ratios.items():
        m = interval_metrics(df.ref_price.to_numpy() * r, df.lo, df.hi, df.mid)
        m.update(band_metrics(band_of(r), df.true_band.to_numpy()))
        rows.append({"method": name, **m})
    t = pd.DataFrame(rows)
    seeds = t[t.method.str.startswith("CLM seed")]
    if len(seeds) > 1:
        mean = seeds.drop(columns="method").mean().to_dict()
        mean.update(method="CLM seeds (mean)", hit_rate_std=float(seeds.interval_hit_rate.std()))
        t = pd.concat([t, pd.DataFrame([mean])], ignore_index=True)
    return t


def replay_table(rep: pd.DataFrame, offers: pd.DataFrame, train_df: pd.DataFrame, runs: list[dict]) -> pd.DataFrame:
    ratios, _ = method_ratios(rep, train_df, runs, "replay")
    rows = []
    for src in ("sku", "family"):
        mask = (rep.ref_source == src).to_numpy()
        if not mask.any():
            continue
        for name, r in ratios.items():
            groups = rep.loc[mask, KEYS3].assign(reserve=rep.ref_price.to_numpy()[mask] * r[mask])
            rows.append({"reference": src, "method": name, **replay(offers, groups)})
    return pd.DataFrame(rows)


def acceptance_table(df: pd.DataFrame, offers: pd.DataFrame, ratios: dict[str, np.ndarray]) -> pd.DataFrame:
    """Daily acceptance on the labelled days themselves (reserve = market price x method ratio)."""
    rows = []
    for name, r in ratios.items():
        groups = df[KEYS3].assign(reserve=df.ref_price.to_numpy() * r)
        rows.append({"method": name, **replay(offers, groups)})
    return pd.DataFrame(rows)


def breakdowns(df: pd.DataFrame, hits: dict[str, np.ndarray]) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    d = df.copy()
    top = d.manufacturer.value_counts().index[:6]
    d["brand"] = np.where(d.manufacturer.isin(top), d.manufacturer, "other")
    d["price_tier"] = pd.cut(d.ref_price, [0, 150, 350, np.inf], labels=["<$150", "$150-350", ">=$350"])
    d["bid_history"] = pd.cut(d.n_offers_30, [-1, 9, 29, np.inf], labels=["<10 bids/30d", "10-29", "30+"])
    for name, h in hits.items():
        d[name] = h
    out = []
    for dim in ("brand", "price_tier", "bid_history"):
        g = d.groupby(dim, observed=True)
        t = g[list(hits)].mean()
        t.insert(0, "n", g.size())
        t = t.reset_index().rename(columns={dim: "value"})
        t.insert(0, "dimension", dim)
        t["value"] = t["value"].astype(str)
        out.append(t)
    return pd.concat(out, ignore_index=True)


def prediction_examples(df: pd.DataFrame, probs: np.ndarray, n: int = 8, seed: int = 0) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    idx = np.random.default_rng(seed).choice(len(df), size=min(n, len(df)), replace=False)
    rows = []
    for i in idx:
        r, p = df.iloc[i], probs[i]
        b = decode_bands(p[None])[0]
        top = np.argsort(p)[::-1][:3]
        rows.append({"sku": r.sku, "wh": r.wh, "day": r.day.date(), "market_price": round(r.ref_price, 2),
                     "assurant_floor_interval": f"{r.lo:.0f}-{r.hi:.0f}",
                     "clm_reserve": round(r.ref_price * BAND_MIDS[b], 2),
                     "top_bands": ", ".join(f"{BAND_LABELS[k]} ({p[k]:.0%})" for k in top)})
    return pd.DataFrame(rows)


In [ ]:
%%writefile reserve_clm/embed.py
"""Qwen3 last-token embeddings (the reader CLM's heads sit on), cached on disk by text hash.

Recipe matches CLM's train/embed_utils.py: tokens without special tokens, keep the last
max_len - 1, last-token hidden state, L2-normalised. Batches are length-sorted and right
padded; causal attention means real tokens never see the padding.
"""
from __future__ import annotations

import hashlib
import os
import time

import numpy as np


def _key(text: str) -> str:
    return hashlib.sha1(text.encode()).hexdigest()


class QwenEmbedder:
    """The model is loaded only when a text is not in the cache yet, so a re-run whose texts are
    all cached never touches the GPU (useful on persistent storage)."""

    def __init__(self, model_name: str, max_len: int = 2048, batch_size: int = 16,
                 cache_path: str | None = None, device: str | None = None, log=print):
        """device: e.g. "cuda:1" to pin the model to one GPU; default = first GPU, else CPU."""
        self.model_name, self.log, self.device_name = model_name, log, device
        self.max_len, self.batch_size, self.cache_path = max_len, batch_size, cache_path
        self.model = self.tok = None
        self._hidden = None
        self._load_cache(cache_path)

    @property
    def hidden(self) -> int:
        if self._hidden is None:
            from transformers import AutoConfig
            self._hidden = int(AutoConfig.from_pretrained(self.model_name).hidden_size)
        return self._hidden

    def _load_model(self) -> None:
        import torch
        from transformers import AutoModel, AutoTokenizer
        self.log(f"[embed] loading {self.model_name}")
        self.tok = AutoTokenizer.from_pretrained(self.model_name)
        device = torch.device(self.device_name or ("cuda" if torch.cuda.is_available() else "cpu"))
        if device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.get_device_capability(device)[0] >= 8 else torch.float16
        else:
            dtype = torch.float32
        # Whole model on one device: no device_map, so `accelerate` is not required.
        self.model = AutoModel.from_pretrained(self.model_name, torch_dtype=dtype).to(device).eval()
        self._hidden = int(self.model.config.hidden_size)
        self.device = self.model.get_input_embeddings().weight.device
        pad = self.tok.pad_token_id
        self.pad_id = pad if pad is not None else (self.tok.eos_token_id or 0)

    def _load_cache(self, path: str | None) -> None:
        self.cache = {}
        if not path:
            return
        if not os.path.exists(path) and not path.endswith(".npz") and os.path.exists(path + ".npz"):
            path = path + ".npz"
        if os.path.exists(path):
            with np.load(path) as z:
                self.cache = dict(zip(z["keys"].tolist(), z["vecs"]))
            if self.cache:
                self._hidden = int(next(iter(self.cache.values())).shape[0])

    def _save_cache(self) -> None:
        if not self.cache_path or not self.cache:
            return
        os.makedirs(os.path.dirname(self.cache_path) or ".", exist_ok=True)
        keys = list(self.cache)
        np.savez(self.cache_path, keys=np.array(keys), vecs=np.stack([self.cache[k] for k in keys]))

    def _ids(self, text: str) -> list[int]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            ids = self.tok(" ", add_special_tokens=False)["input_ids"]
        limit = max(1, self.max_len - 1)
        return ids[-limit:]

    def embed(self, texts: list[str]) -> np.ndarray:
        keys = [_key(t) for t in texts]
        todo = list(dict.fromkeys(t for t, k in zip(texts, keys) if k not in self.cache))
        self.log(f"[embed] {len(set(keys)) - len(todo):,} texts from cache, {len(todo):,} to embed")
        if todo:
            import torch
            if self.model is None:
                self._load_model()
            ids = [self._ids(t) for t in todo]
            order = sorted(range(len(ids)), key=lambda i: len(ids[i]))
            started, done = time.time(), 0
            with torch.no_grad():
                for b, s in enumerate(range(0, len(order), self.batch_size)):
                    chunk = order[s:s + self.batch_size]
                    width = max(len(ids[i]) for i in chunk)
                    x = torch.full((len(chunk), width), self.pad_id, dtype=torch.long)
                    m = torch.zeros((len(chunk), width), dtype=torch.long)
                    for r, i in enumerate(chunk):
                        x[r, :len(ids[i])] = torch.tensor(ids[i])
                        m[r, :len(ids[i])] = 1
                    h = self.model(input_ids=x.to(self.device), attention_mask=m.to(self.device)).last_hidden_state
                    last = torch.tensor([len(ids[i]) - 1 for i in chunk], device=h.device)
                    v = h[torch.arange(len(chunk), device=h.device), last].float()
                    v = torch.nn.functional.normalize(v, dim=-1).cpu().numpy().astype(np.float16)
                    for i, vec in zip(chunk, v):
                        self.cache[_key(todo[i])] = vec
                    done += len(chunk)
                    if b % 50 == 0 or done == len(todo):
                        rate = done / max(1e-9, time.time() - started)
                        self.log(f"[embed] {done}/{len(todo)} new texts ({rate:.1f}/s)")
            self._save_cache()
        if not keys:
            return np.zeros((0, self.hidden), dtype=np.float32)
        return np.stack([self.cache[k] for k in keys]).astype(np.float32)

    def close(self) -> None:
        """Release the 8B model (GPU memory) once all texts are embedded."""
        if self.model is None:
            return
        self.model = None
        import gc

        import torch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
%%writefile reserve_clm/train.py
"""Fine-tune CLM's projection heads on cached encoder embeddings.

Same objective as CLM's `train/finetune.py --task choice --loss infonce --targets soft`:
bidirectional InfoNCE between states and the band texts, against a soft target
distribution. The 8B encoder never changes; only the two small heads train.
"""
from __future__ import annotations

import copy
import math
import os
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class TrainConfig:
    epochs: int = 60
    batch: int = 256
    lr: float = 5e-4
    weight_decay: float = 0.0
    patience: int = 10
    seed: int = 1234
    width: int = 1536      # used only when there is no compatible warm-start checkpoint
    depth: int = 3
    proj: int = 512


class HeadModel(nn.Module):
    def __init__(self, sh: nn.Module, ah: nn.Module, logit_scale: nn.Parameter):
        super().__init__()
        self.sh, self.ah, self.logit_scale = sh, ah, logit_scale

    def logits(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        zq = F.normalize(self.sh(x), dim=-1)
        zc = F.normalize(self.ah(c), dim=-1)
        return self.logit_scale.exp().clamp(max=100.0) * (zq @ zc.t())


def build_model(hidden: int, cfg: TrainConfig, init_ckpt: str | None = None) -> tuple[HeadModel, dict, bool]:
    from clm.heads import make_head  # fail loudly if the CLM package is missing
    head_cfg = dict(width=cfg.width, depth=cfg.depth, proj=cfg.proj, activation="gelu",
                    layernorm=True, residual=False, hidden=hidden)
    ck = torch.load(init_ckpt, map_location="cpu", weights_only=False) if init_ckpt else None
    warm = False
    if (ck is not None
            and "state_head" in ck
            and "inp.weight" in ck["state_head"]
            and ck["state_head"]["inp.weight"].shape[1] == hidden):
        c0 = ck.get("cfg", {})
        head_cfg.update(width=c0.get("width", cfg.width),
                        depth=c0.get("depth", cfg.depth),
                        proj=c0.get("projection_dim", ck.get("projection_dim", cfg.proj)),
                        activation=c0.get("activation", "gelu"),
                        residual=c0.get("residual", False),
                        layernorm=any(k.startswith("norms.") for k in ck["state_head"]))
        warm = True
    sh, ah = make_head(**head_cfg), make_head(**head_cfg)
    logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))
    if warm:
        sh.load_state_dict(ck["state_head"])
        ah.load_state_dict(ck["action_head"])
        if "logit_scale" in ck:
            with torch.no_grad():
                logit_scale.copy_(torch.as_tensor(ck["logit_scale"]).float())
    return HeadModel(sh, ah, logit_scale), head_cfg, warm


def soft_infonce(lg: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """States -> bands against the soft targets, bands -> states against the states that hold them."""
    fwd = -(t * F.log_softmax(lg, 1)).sum(1).mean()
    w = t.t()
    keep = w.sum(1) > 0
    if keep.any():
        w = w[keep] / w[keep].sum(1, keepdim=True)
        bwd = -(w * F.log_softmax(lg.t()[keep], 1)).sum(1).mean()
        return (fwd + bwd) / 2
    return fwd


def _to_tensor(arr, device: torch.device | str) -> torch.Tensor:
    if isinstance(arr, torch.Tensor):
        return arr.to(device=device, dtype=torch.float32)
    return torch.as_tensor(np.asarray(arr), dtype=torch.float32, device=device)


@torch.no_grad()
def predict_proba(model: HeadModel, x, c, batch: int = 4096) -> np.ndarray:
    device = next(model.parameters()).device
    model.eval()
    ct = _to_tensor(c, device)
    if len(x) == 0:
        return np.zeros((0, len(c)), dtype=np.float32)
    out = [F.softmax(model.logits(_to_tensor(x[i:i + batch], device), ct), dim=1).cpu().numpy()
           for i in range(0, len(x), batch)]
    return np.concatenate(out) if out else np.zeros((0, len(c)), dtype=np.float32)


@dataclass
class TrainResult:
    model: HeadModel
    head_cfg: dict
    warm_start: bool
    best_epoch: int
    best_score: float
    history: list = field(default_factory=list)


def train_heads(x_tr, t_tr, c, x_val, val_score, cfg: TrainConfig, init_ckpt: str | None = None,
                device: str | None = None, log=print) -> TrainResult:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(cfg.seed)
    model, head_cfg, warm = build_model(int(np.asarray(x_tr).shape[1]), cfg, init_ckpt)
    model.to(device)
    x = _to_tensor(x_tr, device)
    t = _to_tensor(t_tr, device)
    ct = _to_tensor(c, device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    steps = math.ceil(len(x) / cfg.batch) * cfg.epochs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, total_steps=max(1, steps),
                                                pct_start=0.1, anneal_strategy="cos")
    gen = torch.Generator().manual_seed(cfg.seed)
    best = val_score(predict_proba(model, x_val, c))
    best_state, best_epoch, stale = copy.deepcopy(model.state_dict()), 0, 0
    history = [{"epoch": 0, "train_loss": None, "val_score": best}]
    log(f"[train] seed {cfg.seed} warm_start={warm} epoch 0 val {best:.4f}")
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        total = 0.0
        for idx in torch.randperm(len(x), generator=gen).split(cfg.batch):
            idx = idx.to(device)
            loss = soft_infonce(model.logits(x[idx], ct), t[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            total += loss.item() * len(idx)
        score = val_score(predict_proba(model, x_val, c))
        history.append({"epoch": epoch, "train_loss": total / len(x), "val_score": score})
        log(f"[train] seed {cfg.seed} epoch {epoch} loss {total / len(x):.4f} val {score:.4f}")
        if score > best + 1e-6:
            best, best_epoch, stale = score, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
            if stale >= cfg.patience:
                log(f"[train] early stop at epoch {epoch} (best {best_epoch})")
                break
    model.load_state_dict(best_state)
    model.eval()
    return TrainResult(model, head_cfg, warm, best_epoch, float(best), history)


def save_checkpoint(res: TrainResult, path: str) -> None:
    """Same layout as CLM checkpoints, so clm.heads.HeadPair / clm.Engine can load it."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    c = res.head_cfg
    torch.save({
        "state_head": res.model.sh.state_dict(),
        "action_head": res.model.ah.state_dict(),
        "logit_scale": res.model.logit_scale.detach().cpu(),
        "cfg": {
            "width": c["width"],
            "depth": c["depth"],
            "projection_dim": c["proj"],
            "activation": c["activation"],
            "layernorm": c["layernorm"],
            "residual": c["residual"],
            "hidden_size": c["hidden"],
            "task": "reserve_band",
        },
        "epoch": res.best_epoch,
        "metrics": {"val_interval_hit_rate": res.best_score},
    }, path)


In [ ]:
%%writefile reserve_clm/pipeline.py
"""End-to-end steps the notebook calls: prepare -> embed -> train -> evaluate."""
from __future__ import annotations

import dataclasses
import json
import os
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from .bands import BAND_KEYS, BAND_MIDS, band_of, soft_targets
from .config import VAL_END
from .data import find_data_files, load_offers, load_products, load_sales
from .evaluate import band_ece, calibration_table, decode_bands, interval_metrics
from .features import build_features
from .labels import assign_split, floor_intervals
from .report import (acceptance_table, breakdowns, method_ratios, prediction_examples, replay_table,
                     score_table)
from .text import band_descriptions, state_text
from .train import TrainConfig, predict_proba, save_checkpoint, train_heads

KEYS3 = ["sku", "wh", "day"]


@dataclass
class RunConfig:
    data_roots: list
    work_dir: str = "work"
    embed_model: str = "Qwen/Qwen3-8B"
    max_len: int = 2048
    embed_batch: int = 16
    seeds: tuple = (1234, 1, 2)
    train: TrainConfig = field(default_factory=TrainConfig)
    replay_max_groups: int = 20000
    init_ckpt: str | None = None
    smoke_rows: int | None = None        # subsample labels per split (quick runs only)
    device: str | None = None            # e.g. "cuda:1"; used for embedding and training (default: first GPU)


@dataclass
class Prepared:
    labels: pd.DataFrame
    replay: pd.DataFrame
    offers: pd.DataFrame
    counts: pd.DataFrame


def prepare(cfg: RunConfig, log=print) -> Prepared:
    files = find_data_files(cfg.data_roots)
    log(f"[data] {files}")
    offers = load_offers(files["offers"])
    sales = load_sales(files["sales"])
    products = load_products(files["products"])
    floors = floor_intervals(offers)
    labels = floors.assign(split=assign_split(floors.day))
    test_groups = offers.loc[offers.day >= VAL_END, KEYS3].drop_duplicates()
    replay = test_groups.sample(n=min(cfg.replay_max_groups, len(test_groups)), random_state=0)
    if cfg.smoke_rows:
        labels = pd.concat([g.sample(min(len(g), cfg.smoke_rows), random_state=0)
                            for _, g in labels.groupby("split")])
        replay = replay.head(cfg.smoke_rows)
    log(f"[data] offers {len(offers):,}  sales lines {len(sales):,}  floor days {len(floors):,}  "
        f"labels used {len(labels):,}  replay groups {len(replay):,}")
    targets = pd.concat([labels[KEYS3], replay[KEYS3]]).drop_duplicates()
    feats = build_features(offers, sales, products, floors, targets)
    labels = labels.merge(feats, on=KEYS3, how="left")
    for c in ("lo", "hi", "mid"):
        labels[f"{c}_ratio"] = labels[c] / labels.ref_price
    labels["true_band"] = band_of(labels.mid_ratio.fillna(1.0))
    priced = (labels.ref_source != "none").to_numpy()
    labels["text"] = ""
    labels.loc[priced, "text"] = [state_text(r) for r in labels[priced].itertuples(index=False)]
    replay = replay.merge(feats, on=KEYS3, how="left")
    replay = replay[replay.ref_source != "none"].reset_index(drop=True)
    replay["text"] = [state_text(r) for r in replay.itertuples(index=False)]
    counts = labels.pivot_table(index="split", columns="ref_source", values="lo", aggfunc="size", fill_value=0)
    for c in ("sku", "family", "none"):
        if c not in counts.columns:
            counts[c] = 0
    counts = counts[["sku", "family", "none"]].reindex(["train", "val", "test"]).fillna(0).astype(int)
    return Prepared(labels.reset_index(drop=True), replay, offers, counts)


def views(prep: Prepared) -> dict[str, pd.DataFrame]:
    L = prep.labels
    main, fam = L.ref_source == "sku", L.ref_source == "family"
    out = {"train": L[main & (L.split == "train")], "val": L[main & (L.split == "val")],
           "test": L[main & (L.split == "test")], "test_fallback": L[fam & (L.split == "test")]}
    return {k: v.reset_index(drop=True) for k, v in out.items()}


def embed_all(prep: Prepared, cfg: RunConfig, embedder=None, log=print):
    from .embed import QwenEmbedder
    V = views(prep)
    band_texts = [band_descriptions()[k] for k in BAND_KEYS]
    texts = list(dict.fromkeys(band_texts + [t for v in V.values() for t in v.text] + list(prep.replay.text)))
    created = embedder is None
    if created:
        slug = cfg.embed_model.replace("/", "_")
        embedder = QwenEmbedder(cfg.embed_model, cfg.max_len, cfg.embed_batch, log=log, device=cfg.device,
                                cache_path=os.path.join(cfg.work_dir, f"embed_cache_{slug}_{cfg.max_len}.npz"))
    log(f"[embed] {len(texts):,} unique texts")
    vecs = embedder.embed(texts)
    index = {t: i for i, t in enumerate(texts)}

    def get(ts):
        return vecs[[index[t] for t in ts]] if len(ts) else np.zeros((0, vecs.shape[1]), dtype=np.float32)

    emb = {k: get(list(v.text)) for k, v in V.items()}
    emb["replay"] = get(list(prep.replay.text))
    emb["bands"] = get(band_texts)
    if created:
        embedder.close()
    return V, emb


def run_training(V: dict, E: dict, cfg: RunConfig, log=print) -> list[dict]:
    tr, val = V["train"], V["val"]
    t_tr = soft_targets(tr.lo_ratio, tr.hi_ratio)

    def val_score(p):
        price = val.ref_price.to_numpy() * BAND_MIDS[decode_bands(p)]
        return interval_metrics(price, val.lo, val.hi, val.mid)["interval_hit_rate"]

    runs = []
    for seed in cfg.seeds:
        res = train_heads(E["train"], t_tr, E["bands"], E["val"], val_score,
                          dataclasses.replace(cfg.train, seed=seed), init_ckpt=cfg.init_ckpt,
                          device=cfg.device, log=log)
        save_checkpoint(res, os.path.join(cfg.work_dir, f"head_seed{seed}.pt"))
        probs = {k: predict_proba(res.model, E[k], E["bands"]) for k in ("val", "test", "test_fallback", "replay")}
        runs.append({"seed": seed, "result": res, "probs": probs})
    return runs


def evaluate_all(prep: Prepared, V: dict, runs: list[dict]) -> dict[str, pd.DataFrame]:
    train_df = V["train"]
    training = pd.DataFrame([{"seed": r["seed"], "warm_start": r["result"].warm_start,
                              "best_epoch": r["result"].best_epoch,
                              "val_interval_hit_rate": r["result"].best_score} for r in runs])
    test, fb = V["test"], V["test_fallback"]
    ratios, ens = method_ratios(test, train_df, runs, "test")
    fb_ratios, _ = method_ratios(fb, train_df, runs, "test_fallback")
    offers_test = prep.offers[prep.offers.day >= VAL_END]

    def hit(df, r):
        p = df.ref_price.to_numpy() * r
        return ((p >= df.lo.to_numpy()) & (p <= df.hi.to_numpy())).astype(float)

    ens_ratio = ratios["CLM ensemble (median band)"]
    return {
        "training": training,
        "headline_test": score_table(test, ratios),
        "headline_fallback": score_table(fb, fb_ratios),
        "acceptance_test": acceptance_table(test, offers_test, ratios),
        "acceptance_fallback": acceptance_table(fb, offers_test, fb_ratios),
        "replay": replay_table(prep.replay, offers_test, train_df, runs),
        "calibration": calibration_table(ens.max(1) if len(ens) else [], hit(test, ens_ratio)),
        "band_ece": pd.DataFrame([{"split": "test", "band_ece": band_ece(ens, test.true_band.to_numpy())}]),
        "breakdowns": breakdowns(test, {"CLM ensemble hit": hit(test, ens_ratio),
                                        "B0 carry-forward hit": hit(test, ratios["B0 carry-forward"])}),
        "examples": prediction_examples(test, ens),
    }


def save_results(results: dict[str, pd.DataFrame], work_dir: str) -> None:
    out = os.path.join(work_dir, "results")
    os.makedirs(out, exist_ok=True)
    for name, t in results.items():
        t.to_csv(os.path.join(out, f"{name}.csv"), index=False)
    head = results["headline_test"].set_index("method").interval_hit_rate.to_dict()
    with open(os.path.join(out, "summary.json"), "w", encoding="utf-8") as f:
        json.dump({"test_interval_hit_rate": head}, f, indent=1)


In [ ]:
#@title 4. Run configuration
import pandas as pd
from IPython.display import display
from reserve_clm.pipeline import RunConfig, prepare, views, embed_all, run_training, evaluate_all, save_results
from reserve_clm.train import TrainConfig
from reserve_clm.bands import BAND_KEYS, BAND_LABELS
from reserve_clm.text import band_descriptions
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

GPU = 1                                           # which GPU to use (0, 1, ...); falls back to CPU if there is none
DEVICE = f"cuda:{GPU}" if torch.cuda.device_count() > GPU else ("cuda" if torch.cuda.is_available() else "cpu")
SMOKE = os.environ.get("RESERVE_SMOKE") == "1"   # tiny CPU run used to test the notebook itself
if SMOKE:
    cfg = RunConfig(data_roots=DATA_ROOTS, work_dir=WORK, embed_model="Qwen/Qwen3-0.6B", max_len=512,
                    embed_batch=8, seeds=(1234,), train=TrainConfig(epochs=3, patience=3),
                    replay_max_groups=40, smoke_rows=40, init_ckpt=None, device=DEVICE)
else:
    from clm.heads import download
    cfg = RunConfig(data_roots=DATA_ROOTS, work_dir=WORK, init_ckpt=download(), device=DEVICE)  # released CLM head
print(cfg)
if DEVICE.startswith("cuda"):
    free, total = torch.cuda.mem_get_info(torch.device(DEVICE))
    print(f"{DEVICE}: {torch.cuda.get_device_name(torch.device(DEVICE))}, {free / 2**30:.1f} GB free of {total / 2**30:.1f} GB "
          "(Qwen3-8B needs about 17 GB)")


In [ ]:
#@title 5. Load data, build labels and features
prep = prepare(cfg)
print("Labelled SKU x warehouse x days by split and reference source")
print("(sku = main set, family = fallback set tested separately, none = cannot be priced)")
display(prep.counts)
V = views(prep)
print({k: len(v) for k, v in V.items()}, "| replay groups:", len(prep.replay))


In [ ]:
#@title 6. What CLM reads: one state and the 14 band candidates
print(V["test"].text.iloc[0])
display(pd.DataFrame({"band": BAND_LABELS, "description": [band_descriptions()[k] for k in BAND_KEYS]}))


In [ ]:
#@title 7. Embed every text once with Qwen3-8B (cached to WORK)
V, E = embed_all(prep, cfg)
import gc; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
print({k: v.shape for k, v in E.items()})


In [ ]:
#@title 8. Fine-tune the CLM heads (one run per seed)
runs = run_training(V, E, cfg)


In [ ]:
#@title 9. Evaluate on March-April 2026 and save results
results = evaluate_all(prep, V, runs)
save_results(results, WORK)
print("Training"); display(results["training"])
cols = ["method", "n", "interval_hit_rate", "too_high_rate", "too_low_rate", "mape_vs_mid", "bias_vs_mid",
        "band_exact", "band_within_1", "band_within_3", "band_within_5", "band_mae"]
print("MAIN TEST SET (SKU-level market price)")
display(results["headline_test"].reindex(columns=cols + ["hit_rate_std"]).round(4))
print("FALLBACK TEST SET (family-level market price)")
display(results["headline_fallback"].reindex(columns=cols).round(4))


In [ ]:
#@title 10. Daily acceptance, replay, calibration, breakdowns, examples
acc_cols = ["method", "n_days", "n_offers", "actual_accept_rate", "pred_accept_rate", "agreement",
            "accept_precision", "accept_recall", "day_exact_match", "day_count_mae",
            "accepted_per_day_actual", "accepted_per_day_pred", "lost_sales_rate", "underpricing_rate"]
print("DAILY ACCEPTANCE on labelled test days: reserve = base price for the day, accept every offer with bid >= reserve")
print("Main test set (SKU-level market price)")
display(results["acceptance_test"].reindex(columns=acc_cols).round(4))
print("Fallback test set (family-level market price)")
display(results["acceptance_fallback"].reindex(columns=acc_cols).round(4))
print("Replay on ALL sampled test-period days (including days where every offer was accepted or none was)")
display(results["replay"].reindex(columns=["reference"] + acc_cols).round(4))
print("Calibration of the CLM ensemble (test, main set)")
display(results["calibration"].round(4)); display(results["band_ece"].round(4))
print("Interval hit rate by segment (test, main set)")
display(results["breakdowns"].round(4))
print("Example predictions")
display(results["examples"])
print("All tables saved to", os.path.join(WORK, "results"))


## How to read the results

- **interval_hit_rate** (main metric): share of test days where the predicted reserve lies inside
  Assurant's floor interval [highest bid turned down, lowest bid accepted].
- **too_high / too_low**: reserve above the lowest accepted bid (would have lost that sale) /
  below the highest bid Assurant turned down (would have accepted a bid Assurant did not).
- **mape / bias vs mid**: price error against the interval midpoint; positive bias = reserve too high.
- **band_exact / within_1 / within_3 / within_5 / mae**: share of days whose predicted band is 0, ≤1, ≤3, ≤5
  bands away from the band containing the interval midpoint (each band = 2.5% of market price, so
  ±3 bands ≈ ±7.5% and ±5 bands ≈ ±12.5%).
- **Daily acceptance**: the predicted reserve is that day's base price; every offer that day with bid ≥ reserve is
  accepted. *agreement* = share of offers where we match Assurant's decision; *accept_precision* = of the offers
  we accept, share Assurant accepted; *accept_recall* = of the offers Assurant accepted, share we accept;
  *day_exact_match* = share of days where every offer's decision matches; *day_count_mae* = average gap between
  how many offers we accept and how many Assurant accepted per day.
  *lost_sales_rate* = we reject, Assurant accepted; *underpricing_rate* = we accept, Assurant did not.
- The first daily-acceptance tables use the labelled test days (they have both accepted and turned-down offers);
  the replay table uses a sample of all test-period days.
- **B0 carry-forward** (previous floor) is the strongest rule baseline; CLM must beat it to be useful.
- Fallback rows use a family-level market price and are reported separately; the model never trains on them.
